In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType
import pandas as pd

spark = SparkSession.builder \
    .appName("Pandas UDF Example") \
    .getOrCreate()

# ❗ Tắt Arrow để tránh lỗi trên Windows
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

df = spark.createDataFrame([(1, 2.0), (2, 3.0), (3, 4.0)], ["id", "value"])

@pandas_udf(DoubleType())
def add_one(v: pd.Series) -> pd.Series:
    return v + 1

df.withColumn("value_plus_one", add_one(df["value"])).show()


# Lazy load

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import pandas as pd
from ultralytics import YOLO

# Khởi tạo SparkSession
spark = SparkSession.builder.master("local[1]").appName("Lazy Load YOLO in UDF").getOrCreate()

# Biến toàn cục để giữ mô hình YOLO đã tải
_model = None

# Hàm để lazy load mô hình YOLO
def load_yolo_model():
    global _model
    if _model is None:
        print("Đang tải mô hình YOLO...")
        _model = YOLO("yolov8n.pt")  # Tải mô hình (thay đổi theo mô hình bạn sử dụng)
        print("Mô hình YOLO đã được tải.")
    return _model

# UDF Pandas để thực hiện dự đoán với YOLO
@udf(returnType=StringType())
def predict_yolo(image_path: str) -> str:
    model = load_yolo_model()  # Lazy load mô hình YOLO
    results = model(image_path)  # Dự đoán cho ảnh
    # Trả về kết quả nhận dạng (hoặc thông tin nào đó bạn muốn)
    return str(results.pandas().iloc[0, 0])  # Đây chỉ là ví dụ, bạn có thể tùy chỉnh trả về kết quả

# Tạo DataFrame mẫu
data = [("image1.jpg",), ("image2.jpg",)]
df = spark.createDataFrame(data, ["image_path"])

# Áp dụng UDF để dự đoán trên cột "image_path"
df_with_predictions = df.withColumn("prediction", predict_yolo(df["image_path"]))

# Hiển thị kết quả
df_with_predictions.show(truncate=False)


In [ ]:
3. Giải thích mã nguồn:
_model: Biến toàn cục dùng để lưu mô hình YOLO đã tải. Mô hình này chỉ được tải một lần cho mỗi partition khi UDF được gọi.

load_yolo_model(): Hàm này kiểm tra nếu mô hình chưa được tải (_model is None) thì nó sẽ tải mô hình YOLO từ file (ở đây là yolov8n.pt). Sau khi tải xong, mô hình sẽ được lưu lại trong biến toàn cục _model để tái sử dụng mà không cần tải lại.

predict_yolo(): Đây là UDF Pandas, hàm này nhận vào đường dẫn đến ảnh và trả về kết quả dự đoán. Trong UDF, chúng ta gọi load_yolo_model() để đảm bảo mô hình được tải trước khi thực hiện dự đoán.

df_with_predictions: Dự đoán cho mỗi ảnh trong DataFrame sẽ được lưu vào một cột mới prediction.

4. Sử dụng UDF trong DataFrame Spark
Khi bạn sử dụng UDF trong Spark, nó sẽ được áp dụng trên mỗi phân vùng của DataFrame, vì vậy mô hình YOLO sẽ được tải mỗi khi một phân vùng cần được xử lý.

Với phương pháp lazy loading, mô hình YOLO chỉ được tải một lần trong mỗi phân vùng, giúp tối ưu hóa tài nguyên và tránh việc tải lại mô hình nhiều lần.

5. Chạy trên nhiều phân vùng
Nếu DataFrame của bạn có nhiều phân vùng (partitions), mô hình YOLO sẽ chỉ được tải một lần cho mỗi phân vùng, giúp giảm thiểu chi phí tải lại mô hình. Điều này đặc biệt hữu ích nếu bạn xử lý một lượng lớn dữ liệu.

6. Lưu ý quan trọng:
Đảm bảo tài nguyên: Nếu bạn xử lý dữ liệu lớn hoặc chạy trên một cluster Spark, bạn cần đảm bảo rằng các tài nguyên phần cứng của bạn có đủ bộ nhớ và CPU để xử lý mô hình YOLO.

Tải mô hình với đúng phiên bản PyTorch: Khi chạy trên Spark, bạn cần đảm bảo rằng môi trường của bạn (cả driver và worker nodes) đều cài đặt đúng phiên bản PyTorch và Ultralytics YOLO.